**Version History**

**2025/8/14**
- a few modifications made by yx
- rearranged the structure;
- replace `make_gif`;
- commented the direct replacement of ELM unit;
- add configuration switch - `generate_animation`, which by default equals False;

In [ ]:
import os, sys
import glob
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py as h5
from tqdm import tqdm, trange

In [ ]:
import h5py
from pyproj import Transformer
from scipy.spatial import cKDTree

In [ ]:
# from makegif import make_gif
import cartopy
from herbie import Herbie #[Yi] Herbie is used by Zhi to process HRRR data?
from herbie.toolbox import EasyMap, pc
# yrb_wbd = np.loadtxt('/global/homes/l/lizh142/hrrr/xyz_csv/yrb_wbd.xyz')
naches_wbd = np.loadtxt('/pscratch/sd/x/xiao284/elmbyhuilin/xyz_csv/naches_wbd.xyz')

generate_animation = False

In [ ]:
def read_daymet_h5(filename):
    data = {}
    with h5py.File(filename, 'r') as f:
        for k, v in f.items():
            try:
                data[k] = v[:]
            except TypeError:
                data_t = {}
                for tk, tv in v.items():
                    data_t[tk] = tv[:]
                data[k] = data_t
    return data

In [ ]:
def write_daymet_h5(filename, data):
    with h5py.File(filename, 'w') as f:
        for k, v in data.items():
            try:
                f.create_dataset(k, data=v)
            except TypeError:
                g = f.create_group(k)
                for tk, tv in v.items():
                    g.create_dataset(tk, data=tv)

In [ ]:
def check_keys(data):
    keys = data.keys()
    for key in keys:
        print(key, type(data[key]))
        if isinstance(data[key], dict):
            _keys = data[key].keys()
            for _key in _keys:
                if int(_key) > 5:
                    break
                print('\t', _key, ': type is', type(data[key][_key]))

In [ ]:
plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 100
# plt.rcParams['figure.figsize'] = (10,4)
plt.rcParams['lines.linewidth'] = 1
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

# Get keys and units

In [ ]:
run = '2024-07-14-142626' # '2024-07-08-110357' '2024-07-14-142626' ignition:'2024-08-27-132724' post-fire:'2024-08-27-150810'
                          # fire: '2024-11-20-210107' no fire: '2024-11-20-220109'
#path = f'/pscratch/sd/l/lizh142/elm/ELM_MOSART_CONUS.{run}/run/'
#path = f'/compass/ber200003/zhi/elm/ELM_MOSART_CONUS.{run}/run/'
path = f'/pscratch/sd/x/xiao284/elmbyhuilin/elm/ELM_MOSART_CONUS.{run}/run/'
years = np.arange(2021, 2024)

In [ ]:
f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{years[-1]}-01-01-00000.nc'
# f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.2021-08-05-00000.nc'
data = xr.open_dataset(f)
data

In [ ]:
keys, names, units = [], [], []
for key in list(data.keys()):
    try:
        keys.append(key)
    except:
        keys.append('')
    try:
        names.append(data[key].long_name)
    except:
        names.append('')
    try:
        units.append(data[key].units)
    except:
        units.append('')
df = pd.DataFrame(data={'keys': keys, 'names': names, 'units': units})
# df.to_csv('all_ELM_vars.csv')
df

In [ ]:
# search keyword
search_keyword = 'NH4'
idx = []
for i in range(len(df)):
    if search_keyword in df['keys'][i] or search_keyword in df['names'][i] or search_keyword in df['units'][i]:
        idx.append(i)
df.iloc[idx, :]

# Plot examples (optional)

## spatial plot of soil moisture

In [ ]:
# plot soil moisture
lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
X, Y = np.meshgrid(lon, lat)

fig, axs = plt.subplots(5, 3, figsize=(12,12), subplot_kw={'projection': cartopy.crs.LambertConformal()})
for i, ax in enumerate(axs.flat):
    EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
    ax.add_feature(cartopy.feature.RIVERS)
    art = ax.pcolormesh(X, Y, data['H2OSOI'].values[60, i, :, :], cmap='Spectral_r', vmin=0, vmax=1, transform=cartopy.crs.PlateCarree())
    fig.colorbar(art, label='[mm3/mm3]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
    ax.set_title(f'layer {i+1}: {str(np.around(data.levgrnd.values[i], 3))} m')
    ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
plt.tight_layout()
plt.show()

In [ ]:
# plot soil moisture in a soil column
plt.plot(data['H2OSOI'].values[160, :, 0, 0], -np.arange(len(data['H2OSOI'].values[60, :, 0, 0])), '-x')
plt.show()

## spatial plot of ET and its components

In [ ]:
_keys = ['QVEGT', 'QSOIL', 'QVEGE']

In [ ]:
_names, _units = [], []
for _key in _keys:
    for i, key in enumerate(keys):
        if key == _key:
            _units.append(units[i])#.replace('s', 'd'))
            _names.append(names[i])
pd.DataFrame(data={'keys': _keys, 'names': _names, 'units': _units})

In [ ]:
# test plot 1 time slice
year = 2021
t = 0 # range(365)

f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
data = xr.open_dataset(f)
lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
X, Y = np.meshgrid(lon, lat)

fig, axs = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': cartopy.crs.LambertConformal()})
et = 0

for i, ax in enumerate(axs.flat):
    if i < 3:
        #layer1 - background with river
        EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
        ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
        ax.add_feature(cartopy.feature.RIVERS)
        et += data[_keys[i]].values[t]*86400
        #layer2 - elm var projected
        art = ax.pcolormesh(X, Y, data[_keys[i]].values[t]*86400, cmap='Spectral_r', transform=cartopy.crs.PlateCarree(), vmin=0, vmax=1)
        fig.colorbar(art, label=_units[0], shrink=0.5, extend='max', orientation='vertical', pad=0.02)
        ax.set_title(_keys[i]+'\n'+_names[i]+'\n['+_units[i]+']')
        #layer3 - watershed boundary
        ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
    elif i == 3:
        ax.axis('off')
        continue
    elif i == 4:
        EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
        ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
        ax.add_feature(cartopy.feature.RIVERS)
        art = ax.pcolormesh(X, Y, et, cmap='Spectral_r', transform=cartopy.crs.PlateCarree(), vmin=0, vmax=1)
        fig.colorbar(art, label=_units[0], shrink=0.5, extend='max', orientation='vertical', pad=0.02)
        ax.set_title(f'Total ET\n{year}, Day {t+1}', fontsize=24)
        ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
    elif i == 5:
        EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
        ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
        ax.add_feature(cartopy.feature.RIVERS)
        art = ax.pcolormesh(X, Y, data['EFLX_LH_TOT'].values[t]*0.0345, cmap='Spectral_r', transform=cartopy.crs.PlateCarree(), vmin=0, vmax=1)
        fig.colorbar(art, label=_units[0], shrink=0.5, extend='max', orientation='vertical', pad=0.02)
        ax.set_title(f'Total ET from LH\n{year}, Day {t+1}', fontsize=24)
        ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
plt.tight_layout()

In [ ]:
if generate_animation == True:
    # [note] for three years data, it tooks 2.5h to complete
    for year in years:
        f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
        data = xr.open_dataset(f)
        lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
        X, Y = np.meshgrid(lon, lat)
        
        for t in tqdm(range(365)):
            fig, axs = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': cartopy.crs.LambertConformal()})
            et = 0
            for i, ax in enumerate(axs.flat):
                if i < 3:
                    EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
                    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
                    ax.add_feature(cartopy.feature.RIVERS)
                    et += data[_keys[i]].values[t]*86400
                    art = ax.pcolormesh(X, Y, data[_keys[i]].values[t]*86400, cmap='Spectral_r', transform=cartopy.crs.PlateCarree(), vmin=0, vmax=1)
                    fig.colorbar(art, label=_units[0], shrink=0.5, extend='max', orientation='vertical', pad=0.02)
                    ax.set_title(_keys[i]+'\n'+_names[i]+'\n['+_units[i]+']')
                    ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
                elif i == 3:
                    ax.axis('off')
                    continue
                elif i == 4:
                    EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
                    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
                    ax.add_feature(cartopy.feature.RIVERS)
                    art = ax.pcolormesh(X, Y, et, cmap='Spectral_r', transform=cartopy.crs.PlateCarree(), vmin=0, vmax=1)
                    fig.colorbar(art, label=_units[0], shrink=0.5, extend='max', orientation='vertical', pad=0.02)
                    ax.set_title(f'Total ET\n{year}, Day {t+1}', fontsize=24)
                    ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
                elif i == 5:
                    EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
                    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
                    ax.add_feature(cartopy.feature.RIVERS)
                    art = ax.pcolormesh(X, Y, data['EFLX_LH_TOT'].values[t]*0.0345, cmap='Spectral_r', transform=cartopy.crs.PlateCarree(), vmin=0, vmax=1)
                    fig.colorbar(art, label=_units[0], shrink=0.5, extend='max', orientation='vertical', pad=0.02)
                    ax.set_title(f'Total ET from LH\n{year}, Day {t+1}', fontsize=24)
                    ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
            plt.tight_layout()
            plt.savefig(f'./figs/{year}_{str(t).zfill(4)}.jpg')
            plt.close()
        

In [ ]:
# ffmpeg manual version
# ffmpeg -framerate 10 -i ./figs/2021_%04d.jpg -c:v gif 2021_ET.gif
# ffmpeg -framerate 10 -i ./figs/2022_%04d.jpg -c:v gif 2022_ET.gif
# ffmpeg -framerate 10 -i ./figs/2023_%04d.jpg -c:v gif 2023_ET.gif

import sys
import os

home_dir = os.path.expanduser("~")
my_utils_path = os.path.join(home_dir, 'my_utils')
if my_utils_path not in sys.path:
    sys.path.append(my_utils_path)

from viz import make_gif_ffmpeg

if generate_animation == True:
    make_gif_ffmpeg(image_folder='./figs', fps=10, input_jpg_fname='2021_%04d.jpg', output_gif_fname='2021_ET.gif')
    make_gif_ffmpeg(image_folder='./figs', fps=10, input_jpg_fname='2022_%04d.jpg', output_gif_fname='2022_ET.gif')
    make_gif_ffmpeg(image_folder='./figs', fps=10, input_jpg_fname='2023_%04d.jpg', output_gif_fname='2023_ET.gif')

## calculate spatial mean of 2mT, rain, snowmelt, soil water content, and _key data, for temporal plot

In [ ]:
tsa, rain, snowmelt, theta = np.zeros(len(years)*365), np.zeros(len(years)*365), np.zeros(len(years)*365), np.zeros(len(years)*365)
for year in tqdm(years):
    f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
    data = xr.open_dataset(f)
    for j in range(365):
        tsa[j+365*(year-years[0])] = data['TSA'].values[j, :, :].mean()-273.15 # K -> deg C
        rain[j+365*(year-years[0])] = data['RAIN'].values[j, :, :].mean()*86400 # mm/s -> mm/d
        snowmelt[j+365*(year-years[0])] = data['QSNOMELT'].values[j, :, :].mean()*86400 # mm/s -> mm/d
        theta[j+365*(year-years[0])] = data['H2OSOI'][j, :10, :, :].mean() # mm3/mm3

In [ ]:
fluxes = True # True False

In [ ]:
if fluxes:
    _keys = [#'CWDC_TO_LITR2C', 'CWDC_TO_LITR3C', 
     'LITR1C_TO_SOIL1C', 'LITR2C_TO_SOIL2C', 'LITR3C_TO_SOIL3C', 
     'SOIL1C_TO_SOIL2C', 'SOIL2C_TO_SOIL3C', 'SOIL3C_TO_SOIL4C']
else:
    _keys = ['CWDC', 'LITR1C', 'LITR2C', 'LITR3C', 'SOIL1C', 'SOIL2C', 'SOIL3C', 'SOIL4C']

In [ ]:
_names, _units = [], []
for _key in _keys:
    for i, key in enumerate(keys):
        if key == _key:
            _units.append(units[i])#.replace('/s', '/d'))#.replace('gC', 'mgC')
            _names.append(names[i])
pd.DataFrame(data={'keys': _keys, 'names': _names, 'units': _units})

In [ ]:
#[note] get three year average
d = np.zeros((len(years)*365, len(_keys)))
for year in tqdm(years):
    f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
    data = xr.open_dataset(f)
    for j in range(365):
        for i in range(len(_keys)):
            d[j+365*(year-years[0]), i] = data[_keys[i]].values[j, :, :].mean()#*86400#*1000 # gC/m^2/s -> gC/m^2/d

## temporal plot of all _keys data

In [ ]:
# user configure
_keys = ['QVEGT', 'QSOIL', 'QVEGE']
_names, _units = [], []
for _key in _keys:
    for i, key in enumerate(keys):
        if key == _key:
            _units.append(units[i])
            _names.append(names[i])
pd.DataFrame(data={'keys': _keys, 'names': _names, 'units': _units})

In [ ]:
_units

In [ ]:
d.shape[0]

In [ ]:
plt.figure(figsize=(20, 4))
for i in range(len(_keys)):
    plt.plot(d[:, i], label=_keys[i])
plt.xticks(np.arange(0, d.shape[0], 365), 
           np.arange(0, d.shape[0], 365)//365+years[0], rotation=45)
plt.legend(ncols=4, edgecolor='none', facecolor='none')
#plt.xlim(365*35, d.shape[0])
plt.grid(ls='--')
plt.ylabel(_units[0])

# plt.gca().twinx().plot(tsa, label='tsa', lw=3, color='navy', alpha=0.3)
# plt.ylim(-20, 40)
# plt.ylabel('[deg C]')

# plt.gca().twinx().plot(rain, label='rain', lw=3, color='navy', alpha=0.3)
# plt.ylim(-25, 110)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

# plt.gca().twinx().plot(snowmelt, label='snowmelt', lw=3, color='navy', alpha=0.3)
# plt.ylim(-25, 110)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(20, 4))
plt.plot(theta)
plt.xticks(np.arange(0, d.shape[0], 365), 
           np.arange(0, d.shape[0], 365)//365+years[0], rotation=45)
#plt.xlim(365*35, d.shape[0])
plt.grid(ls='--')
plt.ylabel('[mm3/mm3]')

plt.gca().twinx().plot(tsa, label='tsa', lw=3, color='navy', alpha=0.3)
plt.ylim(-20, 40)
plt.ylabel('[deg C]')

# plt.gca().twinx().plot(rain, label='rain', lw=3, color='navy', alpha=0.3)
# plt.ylim(-25, 110)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

# plt.gca().twinx().plot(snowmelt, label='snowmelt', lw=3, color='navy', alpha=0.3)
# plt.ylim(-25, 110)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

plt.grid()
plt.show()

## calculate typical year data

In [ ]:
t_typ, rain_typ, snowmelt_typ, theta_typ, d_typ = 0, 0, 0, 0, 0
for i in range(len(years)):
    t_typ += tsa[i*365:(i+1)*365]
    rain_typ += rain[i*365:(i+1)*365]
    snowmelt_typ += snowmelt[i*365:(i+1)*365]
    theta_typ += theta[i*365:(i+1)*365]
    d_typ += d[i*365:(i+1)*365, :]
t_typ /= len(years)
rain_typ /= len(years)
snowmelt_typ /= len(years)
theta_typ /= len(years)
d_typ /= len(years)

In [ ]:
np.mean(d_typ, axis=1).shape

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(len(_keys)):
    plt.plot(d_typ[:, i], lw=2, label=_keys[i])
#plt.plot(np.mean(d_typ, axis=1), lw=3, c='k')
plt.xticks(np.linspace(0, 364, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan'])
plt.legend(ncols=4, edgecolor='none', facecolor='none')
plt.grid(ls='--')
plt.ylabel(_units[0])
# plt.ylim(0, 160)

# plt.gca().twinx().plot(t_typ, label='tsa', lw=8, color='navy', alpha=0.3)
# plt.ylim(-15, 30)
# plt.ylabel('[deg C]')

# plt.gca().twinx().plot(rain_typ, label='rain', lw=3, color='navy', alpha=0.3)
# plt.ylim(-1, 6)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

# plt.gca().twinx().plot(snowmelt_typ, label='snowmelt', lw=3, color='navy', alpha=0.3)
# plt.ylim(-1, 6)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(theta_typ, lw=2)
plt.xticks(np.linspace(0, 364, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan'])
plt.grid(ls='--')
plt.ylabel(_units[0])
# plt.ylim(0, 160)

plt.gca().twinx().plot(t_typ, label='tsa', lw=8, color='navy', alpha=0.3)
plt.ylim(-8, 23)
# plt.gca().invert_yaxis()
plt.ylabel('[deg C]')

# plt.gca().twinx().plot(rain_typ, label='rain', lw=3, color='navy', alpha=0.3)
# plt.ylim(-1, 6)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

# plt.gca().twinx().plot(snowmelt_typ, label='snowmelt', lw=3, color='navy', alpha=0.3)
# plt.ylim(-1, 6)
# plt.gca().invert_yaxis()
# plt.ylabel('[mm/d]')

plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
total_flux = np.sum(d_typ, axis=1)
plt.plot(total_flux, lw=2)
plt.xticks(np.linspace(0, 364, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan'])
plt.grid(ls='--')
plt.ylabel(_units[0])
# plt.ylim(0, 160)

plt.gca().twinx().plot(t_typ, label='tsa', lw=8, color='navy', alpha=0.3)
plt.ylim(-8, 23)
# plt.gca().invert_yaxis()
plt.ylabel('[deg C]')

plt.grid()
plt.show()

## make daily images and GIFs

In [ ]:
# test plot 1 slice
year = 2021

f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
data = xr.open_dataset(f)
lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
X, Y = np.meshgrid(lon, lat)

if year == years[0]:
    keys, names, units = [], [], []
    for key in list(data.keys()):
        try:
            keys.append(key)
        except:
            keys.append('')
        try:
            names.append(data[key].long_name)
        except:
            names.append('')
        try:
            units.append(data[key].units)
        except:
            units.append('')

    _keys = ['CWDC_TO_LITR2C', 'CWDC_TO_LITR3C', '',
     'LITR1C_TO_SOIL1C', 'LITR2C_TO_SOIL2C', 'LITR3C_TO_SOIL3C', 
     'SOIL1C_TO_SOIL2C', 'SOIL2C_TO_SOIL3C', 'SOIL3C_TO_SOIL4C']
    _names, _units = [], []
    for j, _key in enumerate(_keys):
        if j == 2:
            _units.append('')
            _names.append('')
        for i, key in enumerate(keys):
            if key == _key:
                _units.append(units[i].replace('gC', 'mgC').replace('/s', '/d'))
                _names.append(names[i])

t=0 #range(365)
fig, axs = plt.subplots(3, 3, figsize=(15, 15), subplot_kw={'projection': cartopy.crs.LambertConformal()})
for i, ax in enumerate(axs.flat):
    if i == 2:
        ax.set_title(f'{year}\nDay {t+1}', fontsize=36)
        ax.axis('off')
        continue
    EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
    ax.add_feature(cartopy.feature.RIVERS)
    art = ax.pcolormesh(X, Y, data[_keys[i]].values[t]*1000*86400, cmap='Spectral_r', vmin=0, vmax=300, transform=cartopy.crs.PlateCarree())
    fig.colorbar(art, label='[mgC/m^2/d]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
    ax.set_title('\n\n\n'+_keys[i]+'\n'+_names[i])#+'\n['+_units[i]+']')
    ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
plt.tight_layout()
# plt.savefig(f'./figs/{year}_{str(t).zfill(4)}.jpg')
# plt.close()

In [ ]:
if generate_animation == True:
    for year in tqdm(years):
        f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
        data = xr.open_dataset(f)
        lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
        X, Y = np.meshgrid(lon, lat)
        
        if year == years[0]:
            keys, names, units = [], [], []
            for key in list(data.keys()):
                try:
                    keys.append(key)
                except:
                    keys.append('')
                try:
                    names.append(data[key].long_name)
                except:
                    names.append('')
                try:
                    units.append(data[key].units)
                except:
                    units.append('')
    
            _keys = ['CWDC_TO_LITR2C', 'CWDC_TO_LITR3C', '',
             'LITR1C_TO_SOIL1C', 'LITR2C_TO_SOIL2C', 'LITR3C_TO_SOIL3C', 
             'SOIL1C_TO_SOIL2C', 'SOIL2C_TO_SOIL3C', 'SOIL3C_TO_SOIL4C']
            _names, _units = [], []
            for j, _key in enumerate(_keys):
                if j == 2:
                    _units.append('')
                    _names.append('')
                for i, key in enumerate(keys):
                    if key == _key:
                        _units.append(units[i].replace('gC', 'mgC').replace('/s', '/d'))
                        _names.append(names[i])
    
        for t in tqdm(range(365)):
            fig, axs = plt.subplots(3, 3, figsize=(15, 15), subplot_kw={'projection': cartopy.crs.LambertConformal()})
            for i, ax in enumerate(axs.flat):
                if i == 2:
                    ax.set_title(f'{year}\nDay {t+1}', fontsize=36)
                    ax.axis('off')
                    continue
                EasyMap("10m", ax=ax, crs=cartopy.crs.LambertConformal()).STATES().OCEAN().LAND().COUNTIES().LAKES()
                ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=cartopy.crs.PlateCarree())
                ax.add_feature(cartopy.feature.RIVERS)
                art = ax.pcolormesh(X, Y, data[_keys[i]].values[t]*1000*86400, cmap='Spectral_r', vmin=0, vmax=300, transform=cartopy.crs.PlateCarree())
                fig.colorbar(art, label='[mgC/m^2/d]', shrink=0.5, extend='max', orientation='vertical', pad=0.02)
                ax.set_title('\n\n\n'+_keys[i]+'\n'+_names[i])#+'\n['+_units[i]+']')
                ax.plot(naches_wbd[:,0]+360, naches_wbd[:,1], 'k', lw=2, transform=pc)
            plt.tight_layout()
            plt.savefig(f'./figs_carbon/{year}_{str(t).zfill(4)}.jpg')
            plt.close()
        
        #make_gif(fps=10, gif_fname=f'{year}.gif')

In [ ]:
import sys
import os

home_dir = os.path.expanduser("~")
my_utils_path = os.path.join(home_dir, 'my_utils')
if my_utils_path not in sys.path:
    sys.path.append(my_utils_path)

from viz import make_gif_ffmpeg

if generate_animation == True:
    make_gif_ffmpeg(image_folder='./figs_carbon', fps=10, input_jpg_fname='2021_%04d.jpg', output_gif_fname='2021_carbon.gif')
    make_gif_ffmpeg(image_folder='./figs_carbon', fps=10, input_jpg_fname='2022_%04d.jpg', output_gif_fname='2022_carbon.gif')
    make_gif_ffmpeg(image_folder='./figs_carbon', fps=10, input_jpg_fname='2023_%04d.jpg', output_gif_fname='2023_carbon.gif')

# Export hdf5

## Find x and y

In [ ]:
lon, lat = data.coords['lon'].values-360, data.coords['lat'].values
X, Y = np.meshgrid(lon, lat)

In [ ]:
lonlat = np.zeros((len(X.flatten()), 2))
lonlat[:, 0], lonlat[:, 1] = X.flatten(),  Y.flatten()

In [ ]:
proj_lcc = "+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 +x_0=0 +y_0=0 +datum=WGS84" # daymet crs
proj_wgs84 = "epsg:4326" # latlon
lonlat_to_daymet = np.array(Transformer.from_crs(proj_wgs84, proj_lcc).transform(lonlat[:, 1], lonlat[:, 0]))

In [ ]:
# a, b = np.reshape(lonlat_to_daymet[0], X.shape), np.reshape(lonlat_to_daymet[1], Y.shape)
# area = np.zeros((a.shape[0]-1, a.shape[1]-1))
# for i in range(a.shape[0]-1):
#     for j in range(a.shape[1]-1):
#         area[i, j] = (a[i, j] - a[i+1, j+1]) * (b[i, j] - b[i+1, j+1])
# area.mean()

In [ ]:
# regenerate the mesh in daymet projection coordinate
xv, yv = np.meshgrid(np.linspace(lonlat_to_daymet[0].min(), lonlat_to_daymet[0].max(), 120), 
                     np.linspace(lonlat_to_daymet[1].min(), lonlat_to_daymet[1].max(), 120))

In [ ]:
plt.scatter(lonlat_to_daymet[0], lonlat_to_daymet[1], s=0.5, c='red')
plt.scatter(xv, yv, s=0.5, c='blue')
plt.axis('equal')
plt.show()

In [ ]:
def idw_interpolation(xy_target, xy_source, values, power=2):
    tree = cKDTree(xy_source)
    distances, indices = tree.query(xy_target, k=4)
    weights = 1 / (distances ** power)
    weights /= weights.sum(axis=1, keepdims=True)
    interpolated_values = np.sum(values[indices] * weights, axis=1)
    return interpolated_values

In [ ]:
xy_source = np.zeros((len(lonlat_to_daymet[0]), 2))
xy_source[:, 0], xy_source[:, 1] = lonlat_to_daymet[0], lonlat_to_daymet[1]
xy_target = np.zeros((len(xv.flatten()), 2))
xy_target[:, 0], xy_target[:, 1] = xv.flatten(), yv.flatten()

## Find ELM data

In [ ]:
fluxes = True # True False

In [ ]:
pre_fire = True # True False

In [ ]:
if fluxes:
    _keys = [#'CWDC_TO_LITR2C', 'CWDC_TO_LITR3C', 
     'LITR1C_TO_SOIL1C', 'LITR2C_TO_SOIL2C', 'LITR3C_TO_SOIL3C', 
     'SOIL1C_TO_SOIL2C', 'SOIL2C_TO_SOIL3C', 'SOIL3C_TO_SOIL4C']
else:
    _keys = ['CWDC', 'LITR1C', 'LITR2C', 'LITR3C', 'SOIL1C', 'SOIL2C', 'SOIL3C', 'SOIL4C']

In [ ]:
if fluxes:
    elm_data = {
        "DOC production latlon [molC m^-2 s^-1]": {},
        "DOC production [molC m^-2 s^-1]": {},
        "time [s]": (np.arange(len(years)*365)+365*(years[0]-1980))*86400,
        "x [m]": xv[0,:],
        "y [m]": yv[:,0],
    }
else:
    elm_data = {
        "time [s]": (np.arange(len(years)*365)+365*(years[0]-1980))*86400,
        "x [m]": xv[0,:],
        "y [m]": yv[:,0],
    }
    for _key in _keys:
        elm_data[_key+' [molC m^-2]'] = {}
        elm_data[_key+' latlon [molC m^-2]'] = {}

In [ ]:
if pre_fire:
    for year in tqdm(years):
        f = path+f'ELM_MOSART_CONUS.{run}.elm.h0.{year}-01-01-00000.nc'
        data = xr.open_dataset(f)
        for day in range(365):
            label = str(day+365*(year-years[0]))
            if fluxes:
                f_DOM, fdom = 0.01, '001'
                elm_data['DOC production latlon [molC m^-2 s^-1]'][label] = 0
                theta = data['H2OSOI'].values[day, :10, :, :].mean(axis=0)
                theta[np.isnan(theta)] = 0 
                assert(len(np.unique(np.isnan(theta))) == 1)
                for i, _key in enumerate(_keys):
                    assert(len(np.unique(np.isnan(elm_data['DOC production latlon [molC m^-2 s^-1]'][label]))) == 1)
                    elm_data['DOC production latlon [molC m^-2 s^-1]'][label] += data[_key].values[day, :, :]*f_DOM*theta/12#*area.mean()
                elm_data['DOC production latlon [molC m^-2 s^-1]'][label] /= len(_keys)
                values = elm_data['DOC production latlon [molC m^-2 s^-1]'][label].flatten()
                interpolated_values = idw_interpolation(xy_target, xy_source, values)
                elm_data['DOC production [molC m^-2 s^-1]'][label] = interpolated_values.reshape(120, 120)
            else:
                for i, _key in enumerate(_keys):
                    elm_data[f'{_key} latlon [molC m^-2]'][label] = data[_key].values[day, :, :]
                    values = elm_data[f'{_key} latlon [molC m^-2]'][label].flatten()
                    interpolated_values = idw_interpolation(xy_target, xy_source, values)
                    elm_data[f'{_key} [molC m^-2]'][label] = interpolated_values.reshape(120, 120)

In [ ]:
if not pre_fire:
    run = '2024-11-20-220109' # post-fire:'2024-08-27-150810'
                              # fire: '2024-11-20-210107' no fire: '2024-11-20-220109'
    path = f'/pscratch/sd/l/lizh142/elm/ELM_MOSART_CONUS.{run}/run/'
    fl = sorted(glob.glob(path+f'ELM_MOSART_CONUS.{run}.elm.h0.*.nc'))
    for i, f in enumerate(tqdm(fl)):
        data = xr.open_dataset(f)
        label = str(i)
        if fluxes:
            f_DOM, fdom = 0.01, '001'
            elm_data['DOC production latlon [molC m^-2 s^-1]'][label] = 0
            theta = data['H2OSOI'].values[0, :10, :, :].mean(axis=0)
            theta[np.isnan(theta)] = 0 
            assert(len(np.unique(np.isnan(theta))) == 1)
            for i, _key in enumerate(_keys):
                assert(len(np.unique(np.isnan(elm_data['DOC production latlon [molC m^-2 s^-1]'][label]))) == 1)
                elm_data['DOC production latlon [molC m^-2 s^-1]'][label] += data[_key].values[0, :, :]*f_DOM*theta/12#*area.mean()
            elm_data['DOC production latlon [molC m^-2 s^-1]'][label] /= len(_keys)
            values = elm_data['DOC production latlon [molC m^-2 s^-1]'][label].flatten()
            interpolated_values = idw_interpolation(xy_target, xy_source, values)
            elm_data['DOC production [molC m^-2 s^-1]'][label] = interpolated_values.reshape(120, 120)
        else:
            for i, _key in enumerate(_keys):
                elm_data[f'{_key} latlon [molC m^-2]'][label] = data[_key].values[0, :, :]
                values = elm_data[f'{_key} latlon [molC m^-2]'][label].flatten()
                interpolated_values = idw_interpolation(xy_target, xy_source, values)
                elm_data[f'{_key} [molC m^-2]'][label] = interpolated_values.reshape(120, 120)
    elm_data["time [s]"] = (np.arange(len(fl)) + 15180)*86400

In [ ]:
# loop year: fluxes
output_folder = '/pscratch/sd/x/xiao284/elmbyhuilin/'
if pre_fire:
    if fluxes:
        filename = f'Naches_DOC_{years[0]}_{years[-1]}_fdom{fdom}_{run}.h5'
        filepath = os.path.join(output_folder, filename)
        write_daymet_h5(filepath, elm_data)
    else:
        filename = f'Naches_Cpools_{years[0]}_{years[-1]}_{run}.h5'
        filepath = os.path.join(output_folder, filename)
        write_daymet_h5(filepath, elm_data)
else:
    if fluxes:
        filename = f'Naches_DOC_{years[0]}_{years[-1]}_fdom{fdom}_postfire_{run}.h5'
        filepath = os.path.join(output_folder, filename)
        write_daymet_h5(filepath, elm_data)
    else:
        filename = f'Naches_Cpools_{years[0]}_{years[-1]}_postfire_{run}.h5'
        filepath = os.path.join(output_folder, filename)
        write_daymet_h5(filepath, elm_data)

In [ ]:
# typical year

In [ ]:
typ = np.zeros((365, 
                elm_data['DOC production [molC m^-2 s^-1]']['0'].shape[0], 
                elm_data['DOC production [molC m^-2 s^-1]']['0'].shape[1]))
typ.shape

In [ ]:
for day in trange(365):
    avg = 0
    for year in years:
        label = str(day+365*(year-years[0]))
        avg += elm_data['DOC production [molC m^-2 s^-1]'][label]
    typ[day, :, :] = avg/len(years)

In [ ]:
elm_data_typ = {
    "DOC production [molC m^-2 s^-1]": {},
    "time [s]": (np.arange(len(years)*365)+365*(years[0]-1980))*86400,
    "x [m]": xv[0,:],
    "y [m]": yv[:,0],
}

In [ ]:
for year in tqdm(years):
    for day in range(365):
        label = str(day+365*(year-years[0]))
        elm_data_typ['DOC production [molC m^-2 s^-1]'][label] = typ[day, :, :]

In [ ]:
#write_daymet_h5(f'/home/lizh142/Naches_DOC_typical_{years[0]}_{years[-1]}_fdom{fdom}.h5', elm_data_typ)

output_folder = '/pscratch/sd/x/xiao284/elmbyhuilin/'

filename = f'Naches_DOC_typical_{years[0]}_{years[-1]}_fdom{fdom}.h5'
filepath = os.path.join(output_folder, filename)
write_daymet_h5(filepath, elm_data_typ)